# Detecting evaluation full plotting (0602)

This notebook does **not** scan data automatically. Fill `DATASETS` manually with the result roots you want to plot, then run all cells on af309 with the `xfold-scripts` kernel.


In [ ]:
from pathlib import Path
import math
import re
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

# Fill these paths manually. Each root may be either the timestamp folder
# or the inner detecting.expr* folder.
DATASETS = {
    "expr1timer": [
        # {"host": "c920bn3", "root": Path("/astrum/home/hpchzy/code/data/20260602/c920bn3/outputDetecting/expr1timer/walk20/<timestamp>")},
    ],
    "expr2fsize": [
        # {"host": "c920bn3", "root": Path("/astrum/home/hpchzy/code/data/20260602/c920bn3/outputDetecting/expr2fsize/walk20/<timestamp>")},
    ],
    "expr3interval": [
        # {"host": "c920bn3", "root": Path("/astrum/home/hpchzy/code/data/20260602/c920bn3/outputDetecting/expr3interval/walk20/<timestamp>")},
    ],
    "expr4frkern": [
        # {"host": "c920bn3", "timer": "cntvcto", "root": Path("/astrum/home/hpchzy/code/data/20260602/c920bn3/outputDetecting/expr4frkern/walk20/<timestamp>")},
    ],
}

DROP_TOP_FRAC = 0.01
RL_RH_NQUANT = 1000
RL_RH_DELTA = 5
RL_RH_Q = 900

REPO_ROOT = Path("/astrum/home/hpchzy/code/TacVar")
OUT_ROOT = REPO_ROOT / "scripts_plot/outputDetecting/full0602"
for sub in ["expr1timer", "expr2fsize", "expr3interval", "expr4frkern_tex"]:
    (OUT_ROOT / sub).mkdir(parents=True, exist_ok=True)

TIMER_COLORS = {
    "clock_gettime": "tab:orange",
    "mpi_wtime": "tab:green",
    "tsc": "tab:blue",
    "tsc_asym": "tab:cyan",
    "papi": "tab:red",
    "papix6": "tab:purple",
    "likwid": "tab:brown",
    "cntvct": "tab:cyan",
    "cntvct_fence": "tab:olive",
    "cntvcto": "tab:pink",
}

print("OUT_ROOT =", OUT_ROOT)
print("Configured dataset counts:", {k: len(v) for k, v in DATASETS.items()})


In [ ]:
def parse_meta(path):
    d = {}
    if not path.exists():
        return d
    for line in path.read_text(errors="ignore").splitlines():
        if "=" in line:
            k, v = line.split("=", 1)
            d[k.strip()] = v.strip()
    return d


def read_values(path):
    vals = []
    for line in path.read_text(errors="ignore").splitlines():
        s = line.strip()
        if not s or s.startswith("#"):
            continue
        try:
            vals.append(float(s.replace(",", " ").split()[0]))
        except ValueError:
            pass
    return vals


def q(vals, p):
    vals = sorted(vals)
    if not vals:
        return math.nan
    if len(vals) == 1:
        return vals[0]
    pos = p * (len(vals) - 1)
    lo, hi = math.floor(pos), math.ceil(pos)
    if lo == hi:
        return vals[lo]
    return vals[lo] * (hi - pos) + vals[hi] * (pos - lo)


def empirical_w1(a, b):
    if not a or not b:
        return math.nan
    n = max(len(a), len(b))
    if n == 1:
        return abs(a[0] - b[0])
    return sum(abs(q(a, i / (n - 1)) - q(b, i / (n - 1))) for i in range(n)) / n


def drop_top_pair(measured, theory, frac):
    if frac <= 0 or not measured:
        return list(measured), list(theory)
    pairs = sorted(zip(measured, theory), key=lambda x: x[0])
    keep = max(1, int(math.floor(len(pairs) * (1.0 - frac))))
    pairs = pairs[:keep]
    return [m for m, _ in pairs], [t for _, t in pairs]


def local_w_hat(theory, measured, x_idx, y_idx):
    theory, measured = sorted(theory), sorted(measured)
    n = min(len(theory), len(measured), RL_RH_NQUANT)
    if n <= 1:
        return math.nan, math.nan
    qs = [i / (n - 1) for i in range(n)]
    u = [q(theory, p) for p in qs]
    v = [q(measured, p) for p in qs]
    tau = min(v) - min(u)
    x_idx = max(0, min(x_idx, n - 1))
    y_idx = max(0, min(y_idx, n - 1))
    if y_idx < x_idx:
        return math.nan, math.nan
    num = float(np.mean([abs(v[i] - u[i] - tau) for i in range(x_idx, y_idx + 1)]))
    den = float(np.mean([abs(u[i] - min(u)) for i in range(x_idx, y_idx + 1)]))
    return num, den


def rl_rh_detail(measured, theory):
    low_num, low_den = local_w_hat(theory, measured, 1, RL_RH_Q)
    high_num, high_den = local_w_hat(theory, measured, RL_RH_Q + 1, RL_RH_NQUANT - RL_RH_DELTA - 1)
    rl = 100.0 * low_num / low_den if low_den and not math.isnan(low_den) else math.nan
    rh = 100.0 * high_num / high_den if high_den and not math.isnan(high_den) else math.nan
    return rl, rh, low_num, low_den, high_num, high_den


def find_expr_roots(root):
    root = Path(root)
    if root.name.startswith("detecting.expr"):
        return [root]
    return sorted([p for p in root.rglob("detecting.expr*") if p.is_dir()])


def ta_from_run_dir(run_dir, meta):
    for key in ["ta", "mu_ns", "interval_ns"]:
        if meta.get(key):
            return int(float(meta[key]))
    m = re.search(r"_ta([0-9]+)", run_dir.name)
    if m:
        return int(m.group(1))
    m = re.search(r"tbase([0-9]+)", str(run_dir))
    if m:
        return int(m.group(1))
    return 0


In [ ]:
def collect_dataset(kind, entries):
    rows = []
    for entry in entries:
        host_hint = entry.get("host", "")
        timer_filter = entry.get("timer")
        root = Path(entry["root"])
        for expr_root in find_expr_roots(root):
            for cdf in sorted(expr_root.rglob("*_ta_cdf.csv")):
                run_dir = cdf.parent
                combo_dir = run_dir.parent
                meta = parse_meta(run_dir / "meta.txt") or parse_meta(combo_dir / "meta.txt")
                measured = read_values(cdf)
                if not measured:
                    continue
                timer = meta.get("timer", "")
                if timer_filter and timer != timer_filter:
                    continue
                ta = ta_from_run_dir(run_dir, meta)
                expr_name = meta.get("expr_name", expr_root.name)
                rows.append({
                    "kind": kind,
                    "host": host_hint or meta.get("host", ""),
                    "expr": expr_name,
                    "np": int(float(meta.get("np", 0) or 0)),
                    "timer": timer,
                    "interval_ns": int(float(meta.get("interval_ns", meta.get("mu_ns", ta)) or ta)),
                    "fsize_kib": int(float(meta.get("fsize_kib", 0) or 0)),
                    "fkern": meta.get("fkern", ""),
                    "rkern": meta.get("rkern", ""),
                    "ta": ta,
                    "measured": measured,
                    "theory": [ta] * len(measured),
                    "run_dir": str(run_dir),
                })
    return pd.DataFrame(rows)


def summarize(raw):
    if raw.empty:
        return pd.DataFrame()
    keys = ["kind", "host", "expr", "np", "timer", "interval_ns", "fsize_kib", "fkern", "rkern"]
    out = []
    for key, g in raw.groupby(keys, dropna=False):
        measured_raw = [x for vals in g["measured"] for x in vals]
        theory_raw = [x for vals in g["theory"] for x in vals]
        measured, theory = drop_top_pair(measured_raw, theory_raw, DROP_TOP_FRAC)
        rl, rh, low_num, low_den, high_num, high_den = rl_rh_detail(measured, theory)
        row = dict(zip(keys, key))
        row.update({
            "n_walks": len(g),
            "n_measured_raw": len(measured_raw),
            "n_measured": len(measured),
            "drop_top_frac": DROP_TOP_FRAC,
            "measured_raw_mean": float(np.mean(measured_raw)) if measured_raw else math.nan,
            "measured_raw_max": max(measured_raw) if measured_raw else math.nan,
            "measured_mean": float(np.mean(measured)) if measured else math.nan,
            "measured_q50": q(measured, 0.5),
            "measured_q95": q(measured, 0.95),
            "w1_ns": empirical_w1(measured, theory),
            "rl_pct": rl,
            "rh_pct": rh,
            "low_num": low_num,
            "low_den": low_den,
            "high_num": high_num,
            "high_den": high_den,
        })
        out.append(row)
    return pd.DataFrame(out).sort_values(keys).reset_index(drop=True)


raw_by_kind = {kind: collect_dataset(kind, entries) for kind, entries in DATASETS.items()}
df_by_kind = {kind: summarize(raw) for kind, raw in raw_by_kind.items()}
for kind, df in df_by_kind.items():
    print(kind, "raw rows", len(raw_by_kind[kind]), "summary rows", len(df))
    if not df.empty:
        print(df[["host", "timer", "interval_ns", "fsize_kib", "fkern", "rkern", "n_walks", "w1_ns", "rl_pct", "rh_pct"]].head().to_string(index=False))


In [ ]:
def host_slug(df):
    hosts = sorted([str(h) for h in df["host"].dropna().unique() if str(h)])
    return "_".join(hosts) if hosts else "nohost"


def set_sparse_ticks(ax, values, rotate=False):
    values = sorted(set(int(v) for v in values if not pd.isna(v)))
    if len(values) <= 6:
        ticks = values
    else:
        ticks = values[::2]
        if values[-1] not in ticks:
            ticks.append(values[-1])
    ax.set_xticks(ticks)
    ax.xaxis.set_major_formatter(ScalarFormatter())
    if rotate:
        ax.tick_params(axis="x", labelrotation=35)


def plot_rl_rh(df, xcol, xlabel, out_dir, stem, logx=True):
    if df.empty:
        print(stem, "no data")
        return None
    hosts = list(df["host"].dropna().unique())
    fig, axes = plt.subplots(len(hosts), 2, figsize=(12.5, max(3.2, 3.0 * len(hosts))), squeeze=False, constrained_layout=True)
    for r, host in enumerate(hosts):
        hdf = df[df["host"] == host]
        for c, (metric, ylabel) in enumerate([("rl_pct", "$R_L$ (%)"), ("rh_pct", "$R_H$ (%)")]):
            ax = axes[r][c]
            for timer, g in hdf.groupby("timer"):
                g = g.sort_values(xcol)
                ax.plot(g[xcol], g[metric], marker="o", linewidth=1.5, label=timer, color=TIMER_COLORS.get(timer))
            if logx:
                ax.set_xscale("log", base=2 if xcol == "fsize_kib" else 10)
            set_sparse_ticks(ax, hdf[xcol], rotate=len(set(hdf[xcol])) > 6)
            ax.set_xlabel(xlabel)
            ax.set_ylabel(ylabel)
            ax.set_title(f"{host} {ylabel}")
            ax.grid(True, alpha=0.3)
            ax.legend(fontsize=8)
    out = out_dir / f"{stem}_{host_slug(df)}_drop{DROP_TOP_FRAC:g}.png"
    fig.savefig(out, dpi=220, bbox_inches="tight")
    plt.show()
    print(out)
    return out


plot_rl_rh(df_by_kind["expr2fsize"], "fsize_kib", "fsize (KiB)", OUT_ROOT / "expr2fsize", "expr2fsize_rl_rh")
plot_rl_rh(df_by_kind["expr3interval"], "interval_ns", "interval (ns)", OUT_ROOT / "expr3interval", "expr3interval_rl_rh", logx=True)


In [ ]:
def plot_expr1_timer(raw, out_dir):
    if raw.empty:
        print("expr1timer no data")
        return None
    hosts = list(raw["host"].dropna().unique())
    fig, axes = plt.subplots(len(hosts), 2, figsize=(12.5, max(3.2, 3.0 * len(hosts))), squeeze=False, constrained_layout=True)
    for r, host in enumerate(hosts):
        hraw = raw[raw["host"] == host]
        axh, axc = axes[r]
        for timer, g in hraw.groupby("timer"):
            vals = [x for arr in g["measured"] for x in arr]
            vals, _ = drop_top_pair(vals, [g["ta"].iloc[0]] * len(vals), DROP_TOP_FRAC)
            if not vals:
                continue
            color = TIMER_COLORS.get(timer)
            axh.hist(vals, bins=40, histtype="step", density=True, linewidth=1.4, label=timer, color=color)
            xs = np.sort(vals)
            ys = np.linspace(0, 1, len(xs))
            axc.plot(xs, ys, linewidth=1.4, label=timer, color=color)
        axh.set_title(f"{host} timer histogram")
        axh.set_xlabel("measured time (ns)")
        axh.set_ylabel("density")
        axh.grid(True, alpha=0.3)
        axh.legend(fontsize=8)
        axc.set_title(f"{host} timer CDF")
        axc.set_xlabel("measured time (ns)")
        axc.set_ylabel("CDF")
        axc.grid(True, alpha=0.3)
        axc.legend(fontsize=8)
    out = out_dir / f"expr1timer_hist_cdf_{host_slug(raw)}_drop{DROP_TOP_FRAC:g}.png"
    fig.savefig(out, dpi=220, bbox_inches="tight")
    plt.show()
    print(out)
    return out


plot_expr1_timer(raw_by_kind["expr1timer"], OUT_ROOT / "expr1timer")


In [ ]:
FRKERN_ORDER = ["copy", "add", "scale", "triad", "pow", "dgemm"]


def frkern_to_latex(df, host, timer, caption=None, label=None):
    sub = df[(df["host"] == host) & (df["timer"] == timer)]
    if sub.empty:
        return ""
    caption = caption or f"{host} {timer}: $R_L$ and $R_H$ with different flush kernels (\\%)"
    label = label or f"tab:expr-frkern-{host}-{timer}"
    lines = []
    lines.append(r"\begin{table*}[htbp]")
    lines.append(r"\centering")
    lines.append(rf"\caption{{{caption}}}")
    lines.append(rf"\label{{{label}}}")
    lines.append(r"\begin{tabular*}{\textwidth}{@{\extracolsep{\fill}}lcccccccccccc}")
    lines.append(r"\toprule")
    lines.append(r"\multirow{3}{*}{Rear Flush} & \multicolumn{12}{c}{Front Flush} \\ \cmidrule(lr){2-13}")
    lines.append(r"& \multicolumn{2}{c}{COPY} & \multicolumn{2}{c}{ADD} & \multicolumn{2}{c}{SCALE} & \multicolumn{2}{c}{TRIAD} & \multicolumn{2}{c}{POW} & \multicolumn{2}{c}{DGEMM} \\ \cmidrule(lr){2-3} \cmidrule(lr){4-5} \cmidrule(lr){6-7} \cmidrule(lr){8-9} \cmidrule(lr){10-11} \cmidrule(lr){12-13}")
    lines.append(r"& $R_L$ & $R_H$ & $R_L$ & $R_H$ & $R_L$ & $R_H$ & $R_L$ & $R_H$ & $R_L$ & $R_H$ & $R_L$ & $R_H$ \\ \midrule")
    for rkern in FRKERN_ORDER:
        vals = [rkern.upper()]
        for fkern in FRKERN_ORDER:
            row = sub[(sub["rkern"] == rkern) & (sub["fkern"] == fkern)]
            if row.empty:
                vals.extend(["--", "--"])
            else:
                rr = row.iloc[0]
                vals.extend([f"{rr['rl_pct']:.1f}", f"{rr['rh_pct']:.1f}"])
        lines.append(" & ".join(vals) + r" \\")
    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular*}")
    lines.append(r"\end{table*}")
    return "\n".join(lines)


df4 = df_by_kind["expr4frkern"]
if df4.empty:
    print("expr4frkern no data")
else:
    for host in sorted(df4["host"].dropna().unique()):
        for timer in sorted(df4[df4["host"] == host]["timer"].dropna().unique()):
            tex = frkern_to_latex(df4, host, timer)
            if not tex:
                continue
            out = OUT_ROOT / "expr4frkern_tex" / f"expr4frkern_{host}_{timer}_drop{DROP_TOP_FRAC:g}.tex"
            out.write_text(tex)
            print(out)
            print(tex[:500])
